In [2]:
from industria.data import load_raw
from industria.features import add_features
from scipy import stats
import pandas as pd

df = add_features(load_raw())
df.shape

(10000, 17)

In [5]:
tabela = pd.crosstab(df["Type"], df["Machine failure"])
print(tabela)

Machine failure     0    1
Type                      
H                 982   21
L                5765  235
M                2914   83


In [6]:
chi2, p_valor, gl, esperado = stats.chi2_contingency(tabela)
print("qui-quadrado: ",round(chi2,3))
print("p-valor: ", p_valor)

qui-quadrado:  13.752
p-valor:  0.0010324110359454081


In [7]:
import statsmodels.api as sm


X_inf = df[["Torque [Nm]", "Tool wear [min]", "temp_diff"]].copy()
X_inf.columns = ["torque", "desgaste", "temp_diff"]

X_inf = sm.add_constant(X_inf)     
y_inf = df["Machine failure"]

modelo_logit = sm.Logit(y_inf, X_inf).fit()
print(modelo_logit.summary())

Optimization terminated successfully.
         Current function value: 0.116373
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:        Machine failure   No. Observations:                10000
Model:                          Logit   Df Residuals:                     9996
Method:                           MLE   Df Model:                            3
Date:                Mon, 21 Sep 2026   Pseudo R-squ.:                  0.2139
Time:                        12:27:10   Log-Likelihood:                -1163.7
converged:                       True   LL-Null:                       -1480.5
Covariance Type:            nonrobust   LLR p-value:                5.529e-137
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -3.3887      0.670     -5.060      0.000      -4.701      -2.076
torque         0.1193      0.

In [8]:
import numpy as np
import pandas as pd

odds = np.exp(modelo_logit.params)
ic = np.exp(modelo_logit.conf_int())

resumo = pd.DataFrame({
    "odds_ratio": odds,
    "IC_baixo": ic[0],
    "IC_alto": ic[1],
})
print(resumo.round(3))

           odds_ratio  IC_baixo  IC_alto
const           0.034     0.009    0.125
torque          1.127     1.112    1.141
desgaste        1.011     1.009    1.013
temp_diff       0.502     0.444    0.568


In [9]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

x_vif = df[["Torque [Nm]", "Rotational speed [rpm]", "Tool wear [min]",
            "power", "strain", "temp_diff"]]
x_vif.columns = ["torque", "rotacao", "desgaste", "power", "strain", "temp_diff"]

vif = pd.DataFrame({
    "feature": x_vif.columns,
    "VIF": [variance_inflation_factor(x_vif.values, i)for i in range(x_vif.shape[1])]
})

print(vif.round(2))

     feature    VIF
0     torque  51.68
1    rotacao   5.80
2   desgaste  17.09
3      power  32.44
4     strain  19.92
5  temp_diff   1.00
